# ML-09 — Validation and Research Claim Audit

This notebook audits the FlyRank research paper findings and my own Week-5 K-Means clustering model using the methodology taught in the Week-6 session.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding A — The Freshness Multiplier (Finding #4)

**What the paper says:**

> "The most dramatic finding: 365+ day content that was refreshed within 30 days shows 3.2× health boost (from 10.7 to 34.5) and 57× more impressions (from 71 to 4,039)."
>
> The 31–90 day freshness window is reported as the strongest stable band at a 7.88 : 1 growth-to-decline ratio.

**My methodology question — Where does the label come from?**

The "health score" is a FlyRank composite: impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts). The trend direction label (growing vs declining) is computed from a 30-day-vs-previous-30-day *impression* change. Because impressions contribute 30 of 100 health-score points AND are the basis for trend direction, the "3.2× health boost" and the "57× impression boost" are partially measuring the same underlying signal. A reader might overcount the evidence if they treat health and impressions as two fully independent confirmations of the freshness effect.

This is not a flaw — the paper discloses the health score formula on page 5 — but it is worth noting when evaluating how strong the evidence is.

**My methodology question — Does the validation design carry the claim?**

The paper correctly flags that the 361+ freshness bucket is unstable: it contains 283 growing pages but only 1 declining page, producing the eye-catching 283 : 1 ratio. The 57× impression comparison (71 → 4,039) comes from a similarly small and self-selected group. Without a confidence interval or significance test, we cannot rule out that this specific comparison is driven by a handful of survivor-bias pages — old content that was *already strong enough to justify a refresh*. The paper's own chart-read note warns about this, which is good practice.

A stronger design would control for pre-refresh visibility: did the refreshed 365+ pages already have higher impressions before the update, or did the refresh itself cause the lift? The current comparison is observational and cannot separate these explanations.

### Finding B — Random Forest Feature Importance for Health Score (ML Appendix)

**What the paper says:**

> "Average Position is the #1 predictor of health score at 43% importance, followed by Impressions (32%) and Scroll Depth (15%)."
>
> The paper adds: "importance is descriptive rather than causal" and "health score is partly constructed from some of these inputs."

**My methodology question — Where does the label come from?**

Health score = impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts). Three of the top four features by importance — position (43%), impressions (32%), and scroll depth (15%) — are direct *inputs* to the label formula. Together they account for 90% of feature importance and 80 of 100 label points. The Random Forest is largely re-discovering the scoring formula rather than uncovering an external relationship.

The paper is transparent about this ("importance is descriptive rather than causal"), but the headline "What Predicts Health?" could still lead a reader to treat these as independent drivers of content quality rather than as components of a composite metric.

**My methodology question — Does the validation design carry the claim?**

The methodology section states the RF uses an "80/20 split" but does not specify whether this split is grouped by client. If pages from the same client appear in both train and test, the model can memorize client-level characteristics (similar content strategies, shared age distributions, correlated positions). This would inflate holdout accuracy and make the feature importance ranking look more stable than it would be on truly unseen clients.

Additionally, no p-values or confidence intervals are reported for the importance scores, making it difficult to assess whether the 43% vs 32% gap between position and impressions is robust or within sampling noise.

In [ ]:
# Section 1 is entirely markdown-based methodology analysis.
# The code cell below verifies the health score formula components
# mentioned in the paper to ground the discussion.

print("=" * 60)
print("PAPER METHODOLOGY REFERENCE")
print("=" * 60)
print()
print("Health Score Formula (from paper page 5):")
print("  Impressions:  30 pts")
print("  Position:     30 pts")
print("  CTR:          20 pts")
print("  Scroll Depth: 20 pts")
print("  Total:       100 pts")
print()
print("RF Top Features vs Health Score Components:")
print(f"  {'Feature':<20} {'RF Importance':>15} {'Health Pts':>12} {'Overlap?':>10}")
print(f"  {'-'*20} {'-'*15} {'-'*12} {'-'*10}")

rf_features = [
    ("Average Position", "43%", "30 pts", "YES"),
    ("Impressions",      "32%", "30 pts", "YES"),
    ("Scroll Depth",     "15%", "20 pts", "YES"),
    ("CTR",              "8%",  "20 pts", "YES"),
    ("Clicks",           "2%",  "0 pts",  "no"),
    ("Sessions",         "0%",  "0 pts",  "no"),
    ("Content Age",      "0%",  "0 pts",  "no"),
    ("Word Count",       "0%",  "0 pts",  "no"),
]

for name, imp, pts, overlap in rf_features:
    print(f"  {name:<20} {imp:>15} {pts:>12} {overlap:>10}")

print()
print("Observation: 90% of RF importance comes from features that are")
print("direct inputs to the health score formula (80/100 pts).")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Context

My Week-5 model is a **K-Means clustering** (k=2) on March 2026 warehouse data. In Week 5, I already used `GroupShuffleSplit` by `client_hash_id` (26 train clients, 7 validation clients, 0 overlap).

For this audit, I will show the **before/after** by comparing:
1. **Before (naive random split):** standard `train_test_split` that ignores client grouping
2. **After (grouped split):** the existing `GroupShuffleSplit` from Week 5

This directly demonstrates whether client grouping changes the measured cluster quality.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})"
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connection ready")

In [ ]:
# ── Reproduce the W05 March 2026 feature dataset ──

march_features = con.sql(f"""
WITH march AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(COALESCE(f.gsc_impressions, 0)) AS gsc_impressions,
        SUM(COALESCE(f.gsc_clicks, 0)) AS gsc_clicks,

        AVG(f.gsc_avg_position) AS gsc_avg_position,

        MAX(f.report_date) AS snapshot_date,

        MAX(
            CASE
                WHEN f.gsc_avg_position IS NULL THEN 1
                ELSE 0
            END
        ) AS position_missing

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    ) f

    WHERE f.gsc_data_available IS TRUE

    GROUP BY
        f.client_hash_id,
        f.content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date

    FROM read_parquet(
        '{rel}/dim_content.parquet'
    )
)

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.gsc_impressions,
    m.gsc_clicks,

    CASE
        WHEN m.gsc_impressions > 0
        THEN m.gsc_clicks::DOUBLE / m.gsc_impressions
        ELSE 0
    END AS gsc_ctr,

    m.gsc_avg_position,
    m.position_missing,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        m.snapshot_date
    ) AS stale_days

FROM march m

INNER JOIN content c
    ON m.client_hash_id = c.client_hash_id
    AND m.content_hash_id = c.content_hash_id

WHERE
    c.content_updated_date IS NOT NULL
    AND c.content_updated_date <= m.snapshot_date
""").df()

print("Rows:", len(march_features))
print("Clients:", march_features["client_hash_id"].nunique())
print("Columns:", march_features.columns.tolist())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# ── Feature engineering (same as W05) ──

CLUSTER_FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position",
    "stale_days",
]

def make_cluster_matrix(df):
    X = df[CLUSTER_FEATURES].copy()
    X["gsc_impressions"] = np.log1p(X["gsc_impressions"])
    X["gsc_clicks"] = np.log1p(X["gsc_clicks"])
    return X

FINAL_K = 2

# ══════════════════════════════════════════════════════════
# BEFORE: Naive random split (ignores client grouping)
# ══════════════════════════════════════════════════════════

train_rand, test_rand = train_test_split(
    march_features,
    test_size=0.20,
    random_state=42,
)

X_train_rand_raw = make_cluster_matrix(train_rand)
X_test_rand_raw = make_cluster_matrix(test_rand)

scaler_rand = StandardScaler()
X_train_rand = scaler_rand.fit_transform(X_train_rand_raw)
X_test_rand = scaler_rand.transform(X_test_rand_raw)

km_rand = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
train_labels_rand = km_rand.fit_predict(X_train_rand)
test_labels_rand = km_rand.predict(X_test_rand)

sil_train_rand = silhouette_score(X_train_rand, train_labels_rand)
sil_test_rand = silhouette_score(X_test_rand, test_labels_rand)

# Check client overlap in naive split
rand_train_clients = set(train_rand["client_hash_id"])
rand_test_clients = set(test_rand["client_hash_id"])
rand_overlap = rand_train_clients & rand_test_clients

print("BEFORE — Naive Random Split")
print(f"  Train rows:        {len(train_rand):,}")
print(f"  Test rows:         {len(test_rand):,}")
print(f"  Train clients:     {train_rand['client_hash_id'].nunique()}")
print(f"  Test clients:      {test_rand['client_hash_id'].nunique()}")
print(f"  Client overlap:    {len(rand_overlap)}")
print(f"  Train silhouette:  {sil_train_rand:.4f}")
print(f"  Test silhouette:   {sil_test_rand:.4f}")
print()

In [ ]:
# ══════════════════════════════════════════════════════════
# AFTER: Grouped split by client (same as W05)
# ══════════════════════════════════════════════════════════

groups = march_features["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        march_features,
        groups=groups,
    )
)

train_grouped = march_features.iloc[train_idx].copy()
test_grouped = march_features.iloc[test_idx].copy()

X_train_grp_raw = make_cluster_matrix(train_grouped)
X_test_grp_raw = make_cluster_matrix(test_grouped)

scaler_grp = StandardScaler()
X_train_grp = scaler_grp.fit_transform(X_train_grp_raw)
X_test_grp = scaler_grp.transform(X_test_grp_raw)

km_grp = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
train_labels_grp = km_grp.fit_predict(X_train_grp)
test_labels_grp = km_grp.predict(X_test_grp)

sil_train_grp = silhouette_score(X_train_grp, train_labels_grp)
sil_test_grp = silhouette_score(X_test_grp, test_labels_grp)

print("AFTER — Client-Grouped Split")
print(f"  Train rows:        {len(train_grouped):,}")
print(f"  Test rows:         {len(test_grouped):,}")
print(f"  Train clients:     {train_grouped['client_hash_id'].nunique()}")
print(f"  Test clients:      {test_grouped['client_hash_id'].nunique()}")
print(f"  Client overlap:    {len(set(train_grouped['client_hash_id']) & set(test_grouped['client_hash_id']))}")
print(f"  Train silhouette:  {sil_train_grp:.4f}")
print(f"  Test silhouette:   {sil_test_grp:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════
# COMPARISON TABLE: Before vs After
# ══════════════════════════════════════════════════════════

comparison = pd.DataFrame([
    {
        "split": "Random (before)",
        "train_rows": len(train_rand),
        "test_rows": len(test_rand),
        "client_overlap": len(rand_overlap),
        "train_silhouette": round(sil_train_rand, 4),
        "test_silhouette": round(sil_test_rand, 4),
        "train_test_gap": round(sil_train_rand - sil_test_rand, 4),
    },
    {
        "split": "Grouped (after)",
        "train_rows": len(train_grouped),
        "test_rows": len(test_grouped),
        "client_overlap": 0,
        "train_silhouette": round(sil_train_grp, 4),
        "test_silhouette": round(sil_test_grp, 4),
        "train_test_gap": round(sil_train_grp - sil_test_grp, 4),
    },
])

print("="*60)
print("BEFORE / AFTER COMPARISON")
print("="*60)
display(comparison)

print()
print("Interpretation:")
print(f"  Random split has {len(rand_overlap)} clients appearing in BOTH train and test.")
print(f"  Grouped split has 0 client overlap — the honest question is:")
print(f"  'do the clusters generalize to clients the model never saw?'")
print()
print("  If the grouped-split silhouette is close to the random-split")
print("  silhouette, the structure is real. If there is a large gap,")
print("  the random split was benefiting from client memorization.")

In [ ]:
# ── Cluster profiles under the honest (grouped) split ──

test_grouped_labeled = test_grouped.copy()
test_grouped_labeled["cluster"] = test_labels_grp

# Base rate: how content distributes across clusters
print("BASE RATE — cluster distribution on validation data")
print(test_grouped_labeled["cluster"].value_counts(normalize=True).round(4))
print()

# Profile table
profile = (
    test_grouped_labeled
    .groupby("cluster")
    .agg(
        n=("content_hash_id", "count"),
        impressions_median=("gsc_impressions", "median"),
        impressions_mean=("gsc_impressions", "mean"),
        clicks_median=("gsc_clicks", "median"),
        position_mean=("gsc_avg_position", "mean"),
        stale_days_mean=("stale_days", "mean"),
    )
    .round(2)
)

print("CLUSTER PROFILES — Grouped validation clients")
display(profile)

### Section 2 Summary

The before/after comparison shows the effect of client grouping on measured cluster quality.

- The **random split** allows pages from the same client to appear in both train and test sets. This means the model can benefit from within-client similarity — pages from one client share similar freshness patterns, content strategies, and traffic profiles.
- The **grouped split** ensures that all pages from a given client stay on one side. The validation silhouette score reflects how well the clusters generalize to entirely unseen clients.

The gap between the two silhouette scores is itself a finding about how much client-level memorization was happening in the random split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I will run the full attack checklist from the `hunting-leakage-and-validating` skill on my Week-5 clustering model.

In [ ]:
# ══════════════════════════════════════════════════════════
# LEAKAGE AUDIT — Attack Checklist
# ══════════════════════════════════════════════════════════

print("ATTACK CHECKLIST")
print("=" * 60)
print()

checklist = [
    (
        "Timeline drawn: all features strictly before the label window",
        "PASS",
        "K-Means is unsupervised — there is no label window.\n"
        "         All features (gsc_impressions, gsc_clicks, gsc_ctr,\n"
        "         gsc_avg_position, stale_days) come from the March 2026\n"
        "         warehouse partition. No future months are used."
    ),
    (
        "No label-derived or sibling columns in the features",
        "N/A*",
        "K-Means has no target label, so label leakage does not\n"
        "         apply in the traditional sense. However, gsc_ctr is\n"
        "         computed from gsc_clicks / gsc_impressions — both of\n"
        "         which are also features. This is a derived-feature\n"
        "         concern tested below."
    ),
    (
        "No product flags / existing-system scores as features",
        "PASS",
        "No FlyRank health scores, optimization flags, or\n"
        "         trend_direction are in the feature set."
    ),
    (
        "Split grouped by the repeating entity",
        "PASS",
        "GroupShuffleSplit by client_hash_id. Zero client overlap\n"
        "         between train and test."
    ),
    (
        "Base rate printed next to every metric",
        "PASS",
        "Cluster distribution is shown above. Silhouette scores\n"
        "         are compared across split strategies."
    ),
    (
        "Top feature importance sanity-checked",
        "CHECK",
        "Cluster centroids are inspected below to verify no\n"
        "         single feature dominates suspiciously."
    ),
    (
        "Metrics recomputed out-of-fold, never in-sample",
        "PASS",
        "Validation silhouette is computed on held-out clients\n"
        "         using centroids fitted only on training clients."
    ),
]

for item, status, note in checklist:
    print(f"  [{status:^6}] {item}")
    print(f"         {note}")
    print()

In [ ]:
# ── Derived-feature test: train with vs without gsc_ctr ──
# gsc_ctr = gsc_clicks / gsc_impressions, so it is derived from
# two other features already in the set. Does removing it change
# the cluster structure?

print("DERIVED-FEATURE TEST: gsc_ctr")
print("=" * 60)
print()
print("gsc_ctr = gsc_clicks / gsc_impressions")
print("Both gsc_clicks and gsc_impressions are already features.")
print("Question: does CTR add information, or is it redundant?")
print()

# Features WITHOUT ctr
FEATURES_NO_CTR = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "stale_days",
]

def make_matrix_no_ctr(df):
    X = df[FEATURES_NO_CTR].copy()
    X["gsc_impressions"] = np.log1p(X["gsc_impressions"])
    X["gsc_clicks"] = np.log1p(X["gsc_clicks"])
    return X

X_train_noctr_raw = make_matrix_no_ctr(train_grouped)
X_test_noctr_raw = make_matrix_no_ctr(test_grouped)

scaler_noctr = StandardScaler()
X_train_noctr = scaler_noctr.fit_transform(X_train_noctr_raw)
X_test_noctr = scaler_noctr.transform(X_test_noctr_raw)

km_noctr = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
train_labels_noctr = km_noctr.fit_predict(X_train_noctr)
test_labels_noctr = km_noctr.predict(X_test_noctr)

sil_train_noctr = silhouette_score(X_train_noctr, train_labels_noctr)
sil_test_noctr = silhouette_score(X_test_noctr, test_labels_noctr)

ctr_test = pd.DataFrame([
    {
        "features": "With gsc_ctr (5 features)",
        "train_silhouette": round(sil_train_grp, 4),
        "test_silhouette": round(sil_test_grp, 4),
    },
    {
        "features": "Without gsc_ctr (4 features)",
        "train_silhouette": round(sil_train_noctr, 4),
        "test_silhouette": round(sil_test_noctr, 4),
    },
])

display(ctr_test)

print()
print("Interpretation:")
print("  If silhouette barely changes when CTR is removed, CTR is")
print("  redundant (as expected from a derived feature). This is not")
print("  leakage in the supervised sense, but it does mean the feature")
print("  set carries a partially redundant signal.")

In [ ]:
# ── Inspect cluster centroids for suspicious dominance ──

print("CLUSTER CENTROIDS (standardized scale)")
print("=" * 60)
print()

centroid_df = pd.DataFrame(
    km_grp.cluster_centers_,
    columns=CLUSTER_FEATURES,
)
centroid_df.index.name = "cluster"

display(centroid_df.round(3))

print()
print("Centroid difference (cluster 1 - cluster 0):")
diff = centroid_df.iloc[1] - centroid_df.iloc[0]
for feat, val in diff.items():
    direction = "↑" if val > 0 else "↓"
    print(f"  {feat:<20} {val:+.3f} {direction}")

print()
print("Interpretation:")
print("  If one feature has a centroid difference many times larger")
print("  than the others, it may be dominating the clustering. Check")
print("  whether this makes domain sense or signals a problem.")

In [ ]:
# ── Error examples: pages near the cluster boundary ──

from sklearn.metrics import pairwise_distances

print("FAILURE / BOUNDARY EXAMPLES")
print("=" * 60)
print()

# Distance from each validation page to each centroid
valid_distances = pairwise_distances(
    X_test_grp,
    km_grp.cluster_centers_,
)

test_grouped_audit = test_grouped.copy()
test_grouped_audit["cluster"] = test_labels_grp
test_grouped_audit["dist_to_own"] = [
    valid_distances[i, label]
    for i, label in enumerate(test_labels_grp)
]
test_grouped_audit["dist_to_other"] = [
    valid_distances[i, 1 - label]
    for i, label in enumerate(test_labels_grp)
]
test_grouped_audit["margin"] = (
    test_grouped_audit["dist_to_other"]
    - test_grouped_audit["dist_to_own"]
)

# Pages closest to the boundary (smallest margin)
boundary = test_grouped_audit.nsmallest(10, "margin")

print("Top 10 pages closest to the cluster boundary:")
print("(Small margin = the model is least confident about this assignment)")
print()
display(
    boundary[[
        "client_hash_id", "content_hash_id", "cluster",
        "gsc_impressions", "gsc_clicks", "gsc_avg_position",
        "stale_days", "margin",
    ]].round(3)
)

print()
print("These boundary pages are where the archetype labels are most")
print("fragile. A small change in the data or features could flip them.")
print("This is expected behavior for K-Means and does not indicate an")
print("error — but it limits how confidently we can use cluster labels")
print("for decision-making on individual pages.")

### Section 3 Summary

The leakage audit found:

1. **No label leakage** in the traditional sense — K-Means is unsupervised, so there is no target variable to leak.
2. **Derived-feature redundancy:** `gsc_ctr` is computed from `gsc_clicks / gsc_impressions`, both of which are already in the feature set. The with/without comparison shows whether this redundancy materially affects cluster quality.
3. **No product flags or system scores** in the feature set.
4. **Grouped split is in place** with zero client overlap.
5. **Centroid inspection** reveals which features drive the cluster separation.
6. **Boundary examples** show where the model is least confident — these are the pages where archetype labels should be treated with the most caution.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# ══════════════════════════════════════════════════════════
# CLAIM REWRITE — Before and After
# ══════════════════════════════════════════════════════════

rewrites = [
    {
        "context": "W05 Cluster naming",
        "before": (
            "The K-Means model produces two performance archetypes: "
            "Stale Low-Demand and Active Visible."
        ),
        "after": (
            "In the March 2026 warehouse snapshot, K-Means (k=2) "
            "separated 27,886 content items into two observed groups. "
            "One group was observed to contain pages with higher "
            "stale_days and lower impressions; the other contained "
            "pages with higher impressions and shorter staleness. "
            "These labels are directional descriptions of the "
            "measured feature profiles, not validated content "
            "categories."
        ),
    },
    {
        "context": "W05 Validation claim",
        "before": (
            "The clusters generalize to unseen clients, confirming "
            "that the archetypes are real."
        ),
        "after": (
            "When the model trained on 26 clients was applied to 7 "
            "held-out clients, the silhouette score was observed to "
            "remain in a similar range. This suggests the cluster "
            "structure is not purely driven by client-specific "
            "patterns, but the evidence is limited to one month "
            "of data and one split."
        ),
    },
    {
        "context": "W05 Archetype interpretation",
        "before": (
            "Cluster 0 pages are failing and should be deprioritized."
        ),
        "after": (
            "Cluster 0 pages were measured to have lower impressions "
            "and higher staleness in the March snapshot. This is a "
            "directional signal for decision-support: these pages may "
            "warrant review, but the cluster assignment alone does not "
            "determine whether a page should be kept, refreshed, or "
            "removed."
        ),
    },
]

print("CLAIM REWRITE TABLE")
print("=" * 60)

for i, r in enumerate(rewrites, 1):
    print(f"\n{'─'*60}")
    print(f"Claim {i}: {r['context']}")
    print(f"{'─'*60}")
    print(f"\n  BEFORE (unsafe):")
    print(f"  {r['before']}")
    print(f"\n  AFTER (safe language):")
    print(f"  {r['after']}")

print(f"\n{'═'*60}")
print("Key language shifts:")
print('  "produces" → "separated" (observed, not created)')  
print('  "confirms" → "suggests" (directional, not proven)')  
print('  "are real" → "is not purely driven by" (hedged)')  
print('  "are failing" → "were measured to have lower" (measured)')  
print('  "should be deprioritized" → "may warrant review" (decision-support)')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.